In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import numpy as np
from maite_datasets.image_classification import MNIST

from dataeval import Embeddings, Metadata
from dataeval.config import set_seed
from dataeval.core import ber_mst
from dataeval.data import ClassBalance, ClassFilter, View
from dataeval.extractors import FlattenExtractor

set_seed(42)  # For reproducibility

In [3]:
# Configure the dataset transforms
transforms = [
    lambda x: x / 255.0,  # scale to [0, 1]
    lambda x: x.astype(np.float32),  # convert to float32
]

# Load the MNIST train set and a flattening feature extractor
mnist = MNIST(root="./data/", image_set="train", transforms=transforms, download=True)
extractor = FlattenExtractor()

In [4]:
digits_all = View(mnist, operations=[ClassBalance("interclass", num_samples=6000)])
embeddings_all = Embeddings(digits_all, extractor=extractor, batch_size=64)
labels_all = Metadata(digits_all).class_labels
print("Label counts:", np.unique(labels_all, return_counts=True))

ber_all = ber_mst(embeddings_all, labels_all)
print("Maximum achievable accuracy (10 digits):", 1 - ber_all["upper_bound"])

Label counts: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([600, 600, 600, 600, 600, 600, 600, 600, 600, 600]))


Maximum achievable accuracy (10 digits): 0.9353333333333333


In [5]:
# TEST ASSERTION CELL ###
assert 0.92 < 1 - ber_all["upper_bound"] < 0.95

In [6]:
digits_149 = View(mnist, operations=[ClassFilter((1, 4, 9)), ClassBalance("interclass", num_samples=6000)])
embeddings_149 = Embeddings(digits_149, extractor=extractor, batch_size=64)
labels_149 = Metadata(digits_149).class_labels
print("Label counts:", np.unique(labels_149, return_counts=True))

ber_149 = ber_mst(embeddings_149, labels_149)
print("Maximum achievable accuracy (1, 4, 9):", 1 - ber_149["upper_bound"])

Label counts: (array([1, 4, 9]), array([2000, 2000, 2000]))


Maximum achievable accuracy (1, 4, 9): 0.9725


In [7]:
# TEST ASSERTION CELL ###
assert 0.96 < 1 - ber_149["upper_bound"] < 0.98

In [8]:
labels_binary = labels_149 == 1
print("Label counts:", np.unique(labels_binary, return_counts=True))

ber_binary = ber_mst(embeddings_149, labels_binary)
print("Maximum achievable accuracy (1 vs. not-1):", 1 - ber_binary["upper_bound"])

Label counts: (array([False,  True]), array([4000, 2000]))


Maximum achievable accuracy (1 vs. not-1): 0.9955


In [9]:
# TEST ASSERTION CELL ###
assert 0.99 < 1 - ber_binary["upper_bound"] < 0.998